# 07 - Tensor Products

When you have one qubit, its state is a 2D vector. When you have two qubits, you do not just stack two vectors. You combine them using the tensor product (also called Kronecker product). The result is a 4D vector for 2 qubits, 8D for 3 qubits, 2^n dimensions for n qubits.

This exponential growth is exactly why quantum computers can be powerful -- and why simulating them on classical computers is hard.

In [ ]:
import numpy as np

## Tensor Product of Vectors

If qubit A is in state |a> and qubit B is in state |b>, then the combined 2-qubit system is in state:

`|a> (tensor) |b>` which is written as `|a> |b>` or `|ab>`

In NumPy, use `np.kron()` for the Kronecker (tensor) product.

In [ ]:
ket_0 = np.array([[1], [0]], dtype=complex)
ket_1 = np.array([[0], [1]], dtype=complex)

# |00> = |0> tensor |0>
ket_00 = np.kron(ket_0, ket_0)
print("|00> =", ket_00.flatten())

# |01> = |0> tensor |1>
ket_01 = np.kron(ket_0, ket_1)
print("|01> =", ket_01.flatten())

# |10> = |1> tensor |0>
ket_10 = np.kron(ket_1, ket_0)
print("|10> =", ket_10.flatten())

# |11> = |1> tensor |1>
ket_11 = np.kron(ket_1, ket_1)
print("|11> =", ket_11.flatten())

# These four vectors form the 2-qubit computational basis
# Notice: each is a 4D vector (2^2 = 4)

## How the Tensor Product Works

For vectors [a, b] and [c, d], the tensor product is:

[a*c, a*d, b*c, b*d]

You multiply each element of the first vector by the entire second vector.

In [ ]:
# Manual computation to see how it works
a = np.array([[1], [0]], dtype=complex)  # |0>
b = np.array([[0], [1]], dtype=complex)  # |1>

# |01> = [1*0, 1*1, 0*0, 0*1] = [0, 1, 0, 0]
manual = np.array([[a[0,0]*b[0,0]], 
                   [a[0,0]*b[1,0]], 
                   [a[1,0]*b[0,0]], 
                   [a[1,0]*b[1,0]]])

kron = np.kron(a, b)

print("Manual:", manual.flatten())
print("np.kron:", kron.flatten())
print(f"Match: {np.allclose(manual, kron)}")

## Tensor Product of Matrices

When you apply gate A to qubit 1 and gate B to qubit 2, the combined operation on the 2-qubit system is:

`A (tensor) B`

This gives you a 4x4 matrix from two 2x2 matrices.

In [ ]:
# Identity and Pauli X
I = np.eye(2, dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)

# Apply X to first qubit, I (nothing) to second qubit
X_I = np.kron(X, I)
print("X (tensor) I =")
print(X_I)

# Apply this to |00> -- should flip first qubit to get |10>
ket_00 = np.kron(ket_0, ket_0)
result = X_I @ ket_00
print("\n(X tensor I)|00> =", result.flatten())
print("This is |10>.")

# Apply I to first qubit, X to second qubit
I_X = np.kron(I, X)
result = I_X @ ket_00
print("\n(I tensor X)|00> =", result.flatten())
print("This is |01>.")

## Building Multi-Qubit States

You can build any multi-qubit product state by tensoring single-qubit states.

In [ ]:
# 3-qubit state |010>
ket_010 = np.kron(np.kron(ket_0, ket_1), ket_0)
print("|010> =", ket_010.flatten())
print(f"Dimension: {len(ket_010)} (2^3 = 8)")

# Put first qubit in |+>, rest in |0>
ket_plus = np.array([[1/np.sqrt(2)], [1/np.sqrt(2)]], dtype=complex)
state = np.kron(np.kron(ket_plus, ket_0), ket_0)
print(f"\n|+00> = {state.flatten().round(4)}")
print("This is a superposition of |000> and |100>")

## Entanglement: States That CANNOT Be Factored

This is the big idea. Some 2-qubit states cannot be written as |a> tensor |b> for any single-qubit states |a> and |b>. These states are called entangled.

The Bell state is the most famous entangled state:

`|Bell> = (|00> + |11>) / sqrt(2)`

Try to write this as |a> tensor |b> -- you cannot. The two qubits are correlated in a way that has no classical explanation.

In [ ]:
# Bell state
ket_00 = np.kron(ket_0, ket_0)
ket_11 = np.kron(ket_1, ket_1)

bell = (ket_00 + ket_11) / np.sqrt(2)
print("Bell state |B00> =", bell.flatten().round(4))

# Check probabilities
for i, label in enumerate(['00', '01', '10', '11']):
    prob = abs(bell[i, 0])**2
    print(f"P(|{label}>) = {prob:.4f}")

# 50% chance of |00>, 50% chance of |11>, 0% for |01> or |10>
# If qubit 1 is 0, qubit 2 MUST be 0. If qubit 1 is 1, qubit 2 MUST be 1.
# They are perfectly correlated. This is entanglement.

In [ ]:
# Can we factor the Bell state as |a> tensor |b>?
# If |a> = [a0, a1] and |b> = [b0, b1], then
# |a> tensor |b> = [a0*b0, a0*b1, a1*b0, a1*b1]
#
# For Bell state: [1/sqrt(2), 0, 0, 1/sqrt(2)]
# a0*b0 = 1/sqrt(2)
# a0*b1 = 0
# a1*b0 = 0
# a1*b1 = 1/sqrt(2)
#
# From a0*b1 = 0: either a0=0 or b1=0
# But if a0=0, then a0*b0=0, which contradicts a0*b0 = 1/sqrt(2)
# If b1=0, then a1*b1=0, which contradicts a1*b1 = 1/sqrt(2)
#
# Contradiction. The Bell state CANNOT be factored. It is entangled.

print("The Bell state cannot be written as a tensor product of two single-qubit states.")
print("This is entanglement.")

## Scaling: Why Quantum Computers Are Hard to Simulate

- 1 qubit: 2D vector
- 2 qubits: 4D vector
- 10 qubits: 1,024D vector
- 20 qubits: 1,048,576D vector
- 50 qubits: 1,125,899,906,842,624D vector

Good luck storing that on a classical computer.

In [ ]:
for n in [1, 2, 5, 10, 20, 30, 50]:
    dim = 2**n
    memory_bytes = dim * 16  # complex128 = 16 bytes each
    if memory_bytes < 1024:
        mem_str = f"{memory_bytes} B"
    elif memory_bytes < 1024**2:
        mem_str = f"{memory_bytes/1024:.1f} KB"
    elif memory_bytes < 1024**3:
        mem_str = f"{memory_bytes/1024**2:.1f} MB"
    elif memory_bytes < 1024**4:
        mem_str = f"{memory_bytes/1024**3:.1f} GB"
    else:
        mem_str = f"{memory_bytes/1024**4:.1f} TB"
    print(f"{n:3d} qubits: {dim:>20,} dimensions, ~{mem_str} to store state vector")

## Key Takeaway

- Tensor product combines individual qubit states into a multi-qubit state
- n qubits need a 2^n dimensional vector (exponential growth)
- Tensor product of gates builds multi-qubit operators
- Entangled states cannot be factored as tensor products -- this is what makes quantum computing fundamentally different

## Exercises

1. Build the 2-qubit state |+-> = |+> tensor |->. What are the probabilities of each measurement outcome (|00>, |01>, |10>, |11>)?

2. Build the 4x4 matrix for applying Hadamard to qubit 1 and Pauli Z to qubit 2. Apply it to |00>.

3. Create the other three Bell states: (|00> - |11>)/sqrt(2), (|01> + |10>)/sqrt(2), and (|01> - |10>)/sqrt(2). Verify that all four Bell states are orthogonal to each other.

In [ ]:
# Your code here
